# Language EDA

Exploratory analysis of `language` (~20 values, ~89% English). Examines language distributions,
non-English trends over time, relationships with redaction metrics, and document length patterns.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
PALETTE = "Set2"
DEFAULT_FIGSIZE = (10, 6)
WIDE_FIGSIZE = (14, 5)
HIST_KWARGS = dict(bins=30, edgecolor="black", alpha=0.7)

In [ ]:
DOC_PATH = "../datasets/document_level.csv"
LOOKUP_PATH = "../data/lookup_table.csv"

doc_df = pd.read_csv(DOC_PATH)
lookup_df = pd.read_csv(LOOKUP_PATH)

df = doc_df.merge(
    lookup_df[["folder_name", "language", "year"]],
    on="folder_name",
    how="left",
    suffixes=("", "_lookup"),
)
# Prefer the lookup year if present; fall back to document_level year
if "year_lookup" in df.columns:
    df["year"] = df["year_lookup"].fillna(df["year"])
    df.drop(columns=["year_lookup"], inplace=True)

print(f"document_level shape: {doc_df.shape}")
print(f"merged shape:         {df.shape}")
print(f"unique languages:     {df['language'].nunique()}")
print()
df.head()

## Distribution

In [ ]:
# Bar chart of all languages (sorted by count)
lang_counts = df["language"].value_counts()

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
lang_counts.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[0], edgecolor="black")
ax.set_title("Document Counts by Language")
ax.set_xlabel("Language")
ax.set_ylabel("Number of Documents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart excluding English
non_eng_counts = lang_counts.drop("English", errors="ignore")

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
non_eng_counts.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[1], edgecolor="black")
ax.set_title("Document Counts by Language (Excluding English)")
ax.set_xlabel("Language")
ax.set_ylabel("Number of Documents")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Summary table: counts and percentages
summary = pd.DataFrame({
    "count": lang_counts,
    "pct": (lang_counts / lang_counts.sum() * 100).round(2),
})
summary.index.name = "language"
print(f"Total documents: {lang_counts.sum()}")
print(f"Total languages: {len(lang_counts)}")
print()
summary

## Time Series

In [ ]:
# Non-English documents over time — stacked bar by language
non_eng = df[df["language"] != "English"].copy()
yearly_non_eng = non_eng.groupby(["year", "language"]).size().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(14, 6))
yearly_non_eng.plot.bar(stacked=True, ax=ax, colormap="tab20", edgecolor="black", linewidth=0.3)
ax.set_title("Non-English Documents Over Time (Stacked by Language)")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Documents")
ax.legend(title="Language", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Fraction non-English over time — line plot
yearly_total = df.groupby("year").size()
yearly_non_eng_total = non_eng.groupby("year").size()
frac_non_eng = (yearly_non_eng_total / yearly_total).fillna(0) * 100

fig, ax = plt.subplots(figsize=WIDE_FIGSIZE)
ax.plot(frac_non_eng.index, frac_non_eng.values, marker="o",
        color=sns.color_palette(PALETTE)[2], linewidth=2)
ax.set_title("Non-English Share of Documents Over Time")
ax.set_xlabel("Year")
ax.set_ylabel("Non-English (%)")
plt.tight_layout()
plt.show()

In [ ]:
# Top 5 non-English languages over time — individual lines
top5_non_eng = non_eng_counts.head(5).index.tolist()

fig, ax = plt.subplots(figsize=WIDE_FIGSIZE)
for i, lang in enumerate(top5_non_eng):
    if lang in yearly_non_eng.columns:
        series = yearly_non_eng[lang]
        ax.plot(series.index, series.values, marker="o", markersize=4,
                label=lang, color=sns.color_palette(PALETTE)[i])
ax.set_title("Top 5 Non-English Languages Over Time")
ax.set_xlabel("Year")
ax.set_ylabel("Number of Documents")
ax.legend(title="Language")
plt.tight_layout()
plt.show()

## Redaction Relationship

In [ ]:
# Redaction rate by language
redaction_rate = df.groupby("language")["has_any_redaction"].mean().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
redaction_rate.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[3], edgecolor="black")
ax.set_title("Redaction Rate by Language")
ax.set_xlabel("Language")
ax.set_ylabel("Fraction with Any Redaction")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Redaction rate: English vs non-English
df["is_english"] = df["language"] == "English"
eng_vs_non = df.groupby("is_english")["has_any_redaction"].mean()
eng_vs_non.index = ["Non-English", "English"]

fig, ax = plt.subplots(figsize=(6, 4))
eng_vs_non.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[:2], edgecolor="black")
ax.set_title("Redaction Rate: English vs Non-English")
ax.set_xlabel("")
ax.set_ylabel("Fraction with Any Redaction")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Mean coverage percent by language (for languages with sufficient sample)
redacted = df[df["has_any_redaction"] == True]
lang_sample = redacted["language"].value_counts()
sufficient_langs = lang_sample[lang_sample >= 10].index
mean_cov = (
    redacted[redacted["language"].isin(sufficient_langs)]
    .groupby("language")["mean_coverage_percent"]
    .mean()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
mean_cov.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[4], edgecolor="black")
ax.set_title("Mean Coverage Percent by Language (>=10 Redacted Docs)")
ax.set_xlabel("Language")
ax.set_ylabel("Mean Coverage (%)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## Length Relationship

In [ ]:
# Violin + box plot — total_pages by language (top 6 languages by count)
top6_langs = lang_counts.head(6).index.tolist()
plot_df = df[df["language"].isin(top6_langs)].copy()
plot_df["language"] = pd.Categorical(plot_df["language"], categories=top6_langs, ordered=True)

fig, ax = plt.subplots(figsize=(12, 6))
sns.violinplot(data=plot_df, x="language", y="total_pages",
               palette=PALETTE, inner=None, ax=ax)
sns.boxplot(data=plot_df, x="language", y="total_pages",
            width=0.15, boxprops=dict(facecolor="white", zorder=2),
            ax=ax, fliersize=0)
ax.set_title("Distribution of Total Pages by Language (Top 6)")
ax.set_xlabel("Language")
ax.set_ylabel("Total Pages")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart — median total_pages by language
median_pages = df.groupby("language")["total_pages"].median().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=DEFAULT_FIGSIZE)
median_pages.plot.bar(ax=ax, color=sns.color_palette(PALETTE)[5], edgecolor="black")
ax.set_title("Median Page Count by Language")
ax.set_xlabel("Language")
ax.set_ylabel("Median Total Pages")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# English vs non-English — side-by-side box plot of total_pages
plot_df = df.copy()
plot_df["group"] = plot_df["language"].apply(lambda x: "English" if x == "English" else "Non-English")

fig, ax = plt.subplots(figsize=(8, 5))
sns.violinplot(data=plot_df, x="group", y="total_pages",
               palette=PALETTE, inner=None, ax=ax)
sns.boxplot(data=plot_df, x="group", y="total_pages",
            width=0.15, boxprops=dict(facecolor="white", zorder=2),
            ax=ax, fliersize=0)
ax.set_title("Total Pages: English vs Non-English")
ax.set_xlabel("")
ax.set_ylabel("Total Pages")
plt.tight_layout()
plt.show()